# Step 2: LLM.int8() — 混合精度分解

**目标**：实现 LLM.int8()（Dettmers 2022）的两大支柱——**vector-wise INT8 量化**与**混合精度分解**（把 emergent outlier 维度切到 FP16 相加），作为后续 SmoothQuant 的**对照基准**。理解它"为什么有效但慢"。

**对应 OUTLINE 课时**：1.2 LLM.int8()（~40 分钟）。

> s1 我们看到 emergent outlier 击垮朴素 per-tensor 量化。LLM.int8() 的解法是"分流"：99.9% 正常维度走 INT8，0.1% outlier 维度走 FP16，结果相加。代价是：运行时要在 GPU 上做分支调度 → 慢。这正是 SmoothQuant 要优化的（s3）。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理

LLM.int8() 把一次矩阵乘 `Y = X @ W`（X 激活 `[m,k]`、W 权重 `[k,n]`）拆成两部分：

### (1) Vector-wise INT8（主体，99.9% 计算）

对 X 的每一行 i、W 的每一列 j 各自算一个 scale，用 INT8 精确表示每行/列的相对分布：
$$c_X[i] = \frac{127}{\max_j |X[i,j]|}, \quad X_q[i,:] = \text{round}(X[i,:] \cdot c_X[i])$$
$$c_W[j] = \frac{127}{\max_i |W[i,j]|}, \quad W_q[:,j] = \text{round}(W[:,j] \cdot c_W[j])$$
反量化时把两套 scale 除回去：$\hat{Y}[i,j] = (X_q[i,:] \cdot W_q[:,j]) / (c_X[i] \cdot c_W[j])$。
"vector-wise"= 每个**向量**（行/列）独立 scale，比 per-tensor（全局一个 scale）细得多，所以对正常维度精度极高。

### (2) 混合精度分解（处理 outlier）

但 vector-wise 仍会被 outlier 拉偏。LLM.int8() 再做一步：识别出 X 里那 0.1% 的 outlier 维度（列），把 `X[:, outlier_cols] @ W[outlier_cols, :]` 单独用 **FP16** 算，其余用 INT8 算，两者相加：
$$Y = \underbrace{X_{\text{normal}} @ W_{\text{normal}}}_{\text{INT8}} + \underbrace{X_{\text{outlier}} @ W_{\text{outlier}}}_{\text{FP16}}$$
结果几乎无损，但运行时两套路径 → 延迟上升。**这是 SmoothQuant（s3）要消除的代价**：它用等价变换把 outlier "吸收"进权重，于是整条路径都能走 INT8。

## 本步填空

1. **`vectorwise_absmax_quant(x)`** —— 对 2D 张量做 vector-wise（按行）absmax INT8 量化，返回 `(量化整数张量, 反量化 scale 向量)`。
2. **`mixed_decompose_matmul(x, w, outlier_cols)`** —— **判断/组合型**：把 `x @ w` 拆成 INT8（非 outlier 列）+ FP16（outlier 列）两路相加。outlier 列由 s1 的 `is_emergent_outlier` 逻辑确定（这里直接给 `outlier_cols`）。

In [ ]:
def vectorwise_absmax_quant(x):
    """对 2D 张量做 vector-wise（按行）absmax 对称 INT8 量化。

    参数
    ----
    x : torch.Tensor, 形状 [m, k], float。

    返回
    ----
    (x_q, scale) 元组：
      - x_q : torch.Tensor, 形状 [m, k], dtype torch.int8。每行 round(x * c) 后的整数。
      - scale : torch.Tensor, 形状 [m], float。每行的反量化 scale，含义是
                "原始值 = x_q / scale"（即 c = 127/max|row|）。
                我们存 c 本身，反量化时除以 c。

    vector-wise 含义：对**每一行**独立算 max(|x|)，scale = 127 / max(|row|)。
    逐元素：x_q[i,j] = round(x[i,j] * scale[i])，再 clamp 到 [-127,127] 转 int8。

    提示
    ----
      - max 沿 dim=1（每行一个 max），keepdim=True 方便广播。
      - 防 0 除：max 加一个极小 epsilon（如 1e-12），或用 clamp_min。
      - round 后用 torch.clamp 到 [-127, 127]，再 .to(torch.int8)。
      - scale 形状必须是 [m]（squeeze 掉 keepdim 那一维）。
    """
    # TODO: 实现 vector-wise absmax INT8 量化，返回 (x_q[int8], scale[m])。
    raise NotImplementedError


# 脚手架（提供）：用 vector-wise 量化做一次纯 INT8 matmul（不含混合分解）。
def int8_matmul(x, w):
    """全 INT8 matmul：X 按行量化、W 按列量化，反量化时除两套 scale。"""
    x_q, c_x = vectorwise_absmax_quant(x)
    w_q, c_w = vectorwise_absmax_quant(w.t())   # 转置让 W 的"列"变成"行"再量化
    # INT8 整数 matmul（用 int32 累加防溢出），再除以两套 scale
    acc = x_q.to(torch.int32) @ w_q.to(torch.int32).t()   # [m, n]
    return acc.float() / (c_x.unsqueeze(1) * c_w.unsqueeze(0))

In [ ]:
def mixed_decompose_matmul(x, w, outlier_cols):
    """LLM.int8() 混合精度分解 matmul：outlier 列走 FP16，其余走 INT8，相加。

    参数
    ----
    x : torch.Tensor [m, k] 激活（FP32）。
    w : torch.Tensor [k, n] 权重（FP32）。
    outlier_cols : 长度=k 的 bool 张量（或 int index 张量），标记 x 的哪些**列**（=W 的哪些行）
                   是 emergent outlier。

    返回
    ----
    torch.Tensor [m, n]：近似 Y = x @ w 的结果。
      - 对 x 的**非 outlier 列**与对应 W 行：走 int8_matmul（vector-wise INT8）。
      - 对 x 的 **outlier 列**与对应 W 行：走 FP32 精确 matmul（这里用 FP32 模拟"高精度路径"，
        真实 LLM.int8() 用 FP16，数值含义一致）。
      - 两部分相加得到完整 Y。

    提示（判断型核心）：
      - 把 outlier_cols 转成 index 张量：out_idx = torch.where(outlier_cols)[0]。
      - normal_idx = torch.where(~outlier_cols)[0]（注意 ~ 对 bool 张量取反）。
      - 切片：x[:, out_idx] @ w[out_idx, :] 是 FP32 路径；
              把 x[:, normal_idx] 与 w[normal_idx, :] 喂给 int8_matmul 是 INT8 路径。
      - 返回 fp_part + int8_part。若没有 outlier（out_idx 为空），INT8 路径即全量。
      - 思考：为什么不能直接对整张 x 跑 INT8？——因为 outlier 列的 max 会把 scale 拉爆，
        其余正常列被压扁。分流后 INT8 路径的 scale 只由正常列决定，精度恢复。
    """
    # TODO: 实现 mixed_decompose_matmul。
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_vectorwise_absmax_quant_shape_and_dtype():
    x = torch.tensor([[1.0, -2.0, 4.0],
                      [0.5,  0.5, 0.5]])
    x_q, scale = vectorwise_absmax_quant(x)
    assert x_q.shape == x.shape
    assert x_q.dtype == torch.int8
    assert scale.shape == torch.Size([2])
    # 第 0 行 max=4 → scale=127/4=31.75；第 1 行 max=0.5 → scale=127/0.5=254
    assert torch.allclose(scale, torch.tensor([31.75, 254.0]), atol=1e-3)

def test_vectorwise_absmax_quant_roundtrip():
    torch.manual_seed(0)
    x = torch.randn(8, 16) * 3
    x_q, scale = vectorwise_absmax_quant(x)
    deq = x_q.float() / scale.unsqueeze(1)
    err = (deq - x).abs().mean() / x.abs().mean()
    assert err.item() < 0.02, f"反量化误差应 <2%，实际 {err.item():.4f}"

def test_int8_matmul_close_to_fp():
    torch.manual_seed(1)
    x = torch.randn(4, 8); w = torch.randn(8, 6)
    y_int8 = int8_matmul(x, w)
    y_ref = x @ w
    rel = (y_int8 - y_ref).abs().mean() / y_ref.abs().mean()
    assert rel.item() < 0.03, f"纯 INT8 matmul 相对误差应 <3%，实际 {rel.item():.4f}"

def test_mixed_decompose_handles_outlier_better_than_full_int8():
    torch.manual_seed(2)
    x = torch.randn(4, 16); w = torch.randn(16, 8)
    # 注入一个 30× outlier 列
    x[:, 3] *= 30.0
    outlier_cols = torch.zeros(16, dtype=torch.bool); outlier_cols[3] = True

    y_mixed = mixed_decompose_matmul(x, w, outlier_cols)
    y_ref = x @ w
    err_mixed = (y_mixed - y_ref).abs().mean() / y_ref.abs().mean()

    # 对照：纯 INT8（不分流）误差应明显更大
    y_full_int8 = int8_matmul(x, w)
    err_full = (y_full_int8 - y_ref).abs().mean() / y_ref.abs().mean()
    assert err_mixed.item() < err_full.item() * 0.5, \
        f"混合分解误差 {err_mixed:.4f} 应 < 纯INT8 {err_full:.4f} 的一半"

def test_mixed_decompose_no_outlier_matches_int8():
    x = torch.randn(4, 8); w = torch.randn(8, 6)
    outlier_cols = torch.zeros(8, dtype=torch.bool)   # 无 outlier
    y_mixed = mixed_decompose_matmul(x, w, outlier_cols)
    y_int8 = int8_matmul(x, w)
    assert torch.allclose(y_mixed, y_int8, atol=1e-3)

## L2：tiny 验证（CPU）—— 混合分解误差 < 全 INT8

合成带 outlier 的激活，对比：① 纯 INT8（outlier 拉爆 scale）② 混合分解（outlier 切 FP32）。混合分解应明显更接近 FP32 真值。再在 tiny Qwen2 的一层 FFN 上验证 matmul 管线通。

In [ ]:
# 合成验证：outlier 列使纯 INT8 崩，混合分解救回
torch.manual_seed(42)
m, k, n = 8, 32, 16
x = torch.randn(m, k) * 0.5
w = torch.randn(k, n) * 0.3
x[:, 5] *= 40.0   # emergent outlier 列
x[:, 11] *= 25.0  # 另一个

# 用 s1 同款的判断逻辑标 outlier（幅值 >= 均值 6×）
col_mags = x.abs().mean(dim=0)
outlier_cols = col_mags >= (col_mags.mean() * 6.0)

y_ref = x @ w
y_int8 = int8_matmul(x, w)
y_mixed = mixed_decompose_matmul(x, w, outlier_cols)

rel_int8 = ((y_int8 - y_ref).abs().mean() / y_ref.abs().mean()).item()
rel_mixed = ((y_mixed - y_ref).abs().mean() / y_ref.abs().mean()).item()
print(f"纯 INT8 相对误差: {rel_int8:.4f}（outlier 拉爆 scale → 崩）")
print(f"混合分解 相对误差: {rel_mixed:.4f}（outlier 切 FP32 → 救回）")
print(f"检出 outlier 列: {torch.where(outlier_cols)[0].tolist()}")
assert rel_mixed < rel_int8 * 0.3, "混合分解应远好于纯 INT8"
print("L2a PASS：混合分解误差 << 全 INT8")

# tiny Qwen2 一层 FFN：gate_proj 的 X @ W 管线
from transformers import Qwen2Config, Qwen2ForCausalLM
def make_tiny(vocab=320, hidden=128, inter=256):
    cfg = Qwen2Config(num_hidden_layers=1, hidden_size=hidden, intermediate_size=inter,
        num_attention_heads=4, num_key_value_heads=2, vocab_size=vocab, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()
tiny = make_tiny()
gate = tiny.model.layers[0].mlp.gate_proj
x_act = torch.randn(4, hidden := tiny.config.hidden_size)
w_gate = gate.weight.data.float().t()   # [hidden, inter]
y_ref = x_act @ w_gate
y_int8 = int8_matmul(x_act, w_gate)
rel_int8_tiny = ((y_int8 - y_ref).abs().mean() / y_ref.abs().mean()).item()
# 无 outlier 时混合分解退化为纯 INT8（out_idx 空），二者误差应相等 → 印证标题主张
outlier_cols_none = torch.zeros(w_gate.shape[0], dtype=torch.bool)
y_mixed_tiny = mixed_decompose_matmul(x_act, w_gate, outlier_cols_none)
rel_mixed_tiny = ((y_mixed_tiny - y_ref).abs().mean() / y_ref.abs().mean()).item()
print(f"\ntiny gate_proj 纯 INT8 相对误差: {rel_int8_tiny:.4f}（无 outlier 时应 <3%）")
print(f"tiny gate_proj 混合分解相对误差: {rel_mixed_tiny:.4f}（无 outlier 时 == 纯 INT8）")
assert rel_int8_tiny < 0.03
assert rel_mixed_tiny <= rel_int8_tiny + 1e-6, "无 outlier 时混合分解应 ≤ 纯 INT8（这里两者相等）"
print("L2b PASS：tiny FFN 的混合分解 ≤ 纯 INT8 matmul")

## L3：H200 执行（真 Qwen2.5-0.5B 的 gate_proj）

GPU 守卫。在真 0.5B 的一层 FFN `gate_proj` 上对比：纯 INT8 vs 混合分解（用真激活扫出的 outlier 列）。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path("steps").resolve()) if (pathlib.Path.cwd()/"steps").exists() else "")
    # 复用 s1 的激活收集 + outlier 判断（本 notebook 自包含，重定义精简版）
    def collect_one_activation(model, input_ids, target_substr):
        found = {}
        def hook(name):
            def _h(mod, inp, out):
                if name not in found:
                    found[name] = inp[0].detach().float().cpu()
            return _h
        hs=[]
        for n,m in model.named_modules():
            if isinstance(m, torch.nn.Linear) and target_substr in n:
                hs.append(m.register_forward_hook(hook(n)))
        with torch.no_grad(): model(input_ids)
        for h in hs: h.remove()
        return found
    def per_chan(a):  # 复用 s1 逻辑
        x=a.float().abs().reshape(-1, a.shape[-1]); return x.mean(0)

    tok = AutoTokenizer.from_pretrained(TINY_MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, dtype=torch.float16,
                                                 device_map="auto").eval()
    ids = tok("The future of AI is", return_tensors="pt").input_ids.to(model.device)
    acts = collect_one_activation(model, ids, "layers.0.mlp.gate_proj")
    name = list(acts)[0]; X = acts[name].reshape(-1, acts[name].shape[-1])   # [tok, hidden]
    W = model.model.layers[0].mlp.gate_proj.weight.data.float().cpu()        # [inter, hidden]
    W = W.t()                                                                # [hidden, inter]

    col_mags = per_chan(X)
    outlier_cols = col_mags >= (col_mags.mean() * 6.0)
    print(f"真 0.5B gate_proj：{int(outlier_cols.sum())}/{X.shape[1]} 列是 outlier")

    y_ref = (X @ W); y_int8 = int8_matmul(X, W); y_mixed = mixed_decompose_matmul(X, W, outlier_cols)
    print(f"纯 INT8 相对误差: {((y_int8-y_ref).abs().mean()/y_ref.abs().mean()).item():.4f}")
    print(f"混合分解 相对误差: {((y_mixed-y_ref).abs().mean()/y_ref.abs().mean()).item():.4f}")
    del model; torch.cuda.empty_cache()
else:
    print("跳过 L3：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查

打印对比表：纯 INT8 vs 混合分解 vs FP32 真值的相对误差，直观看到 LLM.int8() 的"分流"价值。

In [ ]:
def report_llmint8():
    """打印 LLM.int8() 三路径误差对比（基于 L2/L3 的合成 + 真模型观察总结）。"""
    torch.manual_seed(7)
    x = torch.randn(16, 64) * 0.5; w = torch.randn(64, 32) * 0.3
    y_ref = x @ w
    print("=== LLM.int8() 路径对比（合成，无 outlier）===")
    y_i = int8_matmul(x, w)
    print(f"  纯 vector-wise INT8 : 相对误差 {((y_i-y_ref).abs().mean()/y_ref.abs().mean()).item():.4f}")
    # 注入 outlier 重跑
    x2 = x.clone(); x2[:, 9] *= 40.0; x2[:, 20] *= 30.0
    y_ref2 = x2 @ w
    y_i2 = int8_matmul(x2, w)
    cm = x2.abs().mean(0); oc = cm >= cm.mean()*6.0
    y_m2 = mixed_decompose_matmul(x2, w, oc)
    print("\n=== 注入 2 个 outlier 列后 ===")
    print(f"  纯 vector-wise INT8 : 相对误差 {((y_i2-y_ref2).abs().mean()/y_ref2.abs().mean()).item():.4f}（崩）")
    print(f"  混合分解(INT8+FP32) : 相对误差 {((y_m2-y_ref2).abs().mean()/y_ref2.abs().mean()).item():.4f}（救回）")
    print("\n结论：LLM.int8() 用分流保精度，但代价是运行时双路径调度 → 慢（见 s3 SmoothQuant）。")

report_llmint8()